# 02 — AI Regime + Scenarios (workflow_update)

**NON_BASELINE_RUN**. Timed subprocesses; S=5000, N=30.

In [ ]:
import subprocess
import time
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "CLAUDE.md").exists():
    ROOT = ROOT.parent
BASE = [
    "--config",
    "configs/base.yaml",
    "--profile",
    "configs/profiles/workflow_update.yaml",
    "--override",
    "configs/provisional/workflow_update_downstream.yaml",
]


def run(name, cmd):
    print("$", " ".join(cmd), flush=True)
    t = time.perf_counter()
    p = subprocess.run(cmd, cwd=ROOT, check=False)
    elapsed = time.perf_counter() - t
    print(f"[{name}] exit={p.returncode} elapsed={elapsed:.2f}s")
    if p.returncode:
        raise RuntimeError(f"{name} failed")
    return elapsed

In [ ]:
timings = {}
timings["regime"] = run("regime", ["uv", "run", "qshield-ai", "regime", *BASE])
timings["scenarios"] = run("scenarios", ["uv", "run", "qshield-ai", "scenarios", *BASE])
print(timings, "total", sum(timings.values()))

In [ ]:
import json

import numpy as np

p = ROOT / "artifacts/dev/scenarios"
m = json.loads((p / "scenario_manifest.json").read_text())
with np.load(p / "stress_scenarios.npz") as z:
    print("shape", z["scenarios"].shape)
print(
    "gate",
    m.get("gate_status"),
    "regime",
    m.get("target_regime"),
    "S",
    m.get("num_scenarios"),
)